In [3]:
from huggingface_hub import snapshot_download
snapshot_download(
    repo_id="floschne/xm3600",
    repo_type="dataset",
    local_dir="/root/.cache/huggingface/datasets/floschne___xm3600"
)

Fetching 38 files:   0%|          | 0/38 [00:00<?, ?it/s]

'/root/.cache/huggingface/datasets/floschne___xm3600'

In [1]:
import os

In [7]:
from IPython.display import FileLink
FileLink('/kaggle/working/multilingual_checkpoints/best_multilingual_bos.pt') # Replace with your file name


/kaggle/working/multilingual_checkpoints/best_multilingual_bos.pt

In [ ]:
# import os
# import shutil

# BASE = "/kaggle/input/datasets/kavitasriram19/mlp-multilingual"
# TARGET = "/kaggle/working/datasets"

# os.makedirs(TARGET, exist_ok=True)

# def merge_dirs(src, dst):
#     for root, dirs, files in os.walk(src):
#         rel = os.path.relpath(root, src)
#         target_root = os.path.join(dst, rel)
#         os.makedirs(target_root, exist_ok=True)

#         for f in files:
#             src_file = os.path.join(root, f)
#             dst_file = os.path.join(target_root, f)

#             if not os.path.exists(dst_file):  # avoid overwrite
#                 shutil.copy2(src_file, dst_file)

# # go through all dataset shards
# for folder in os.listdir(BASE):
#     full_path = os.path.join(BASE, folder)
#     if os.path.isdir(full_path):
#         datasets_path = os.path.join(full_path, "datasets")
#         if os.path.exists(datasets_path):
#             print(f"Merging: {datasets_path}")
#             merge_dirs(datasets_path, TARGET)

# print("Done merging")

# # check
# print(os.listdir(TARGET))

In [ ]:
import json, os, glob

base = "/kaggle/working/datasets"
img_flickr8k = f"{base}/images/flickr8k"
img_flickr30k = f"{base}/images/flickr30k"
img_viet = f"{base}/images/vietnamese"

print("=" * 60)
print("🔍 COMPREHENSIVE DATASET VALIDATION")
print("=" * 60)

errors = []

# ============ 1. TURKISH ============
print("\n--- TURKISH ---")
with open(f"{base}/turkish/tasviret8k_captions.json", "r", encoding="utf-8") as f:
    data = json.load(f)
images = data["images"]
n_imgs = len(images)
n_caps = sum(len(img["sentences"]) for img in images)
print(f"  Images: {n_imgs} | Captions: {n_caps}")

# Check if image files exist in flickr8k
missing = 0
for img in images[:100]:  # sample check
    if not os.path.exists(f"{img_flickr8k}/{img['filename']}"):
        missing += 1
        if missing <= 3:
            print(f"MISSING: {img['filename']}")
if missing == 0:
    print(f"Image check passed (100 samples)")
else:
    errors.append(f"Turkish: {missing}/100 images missing from flickr8k")
    print(f"{missing}/100 sample images missing!")

# ============ 2. ARABIC ============
print("\n--- ARABIC ---")
with open(f"{base}/arabic/Flickr8k.arabic.full.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f.readlines() if l.strip()]
n_caps = len(lines)
filenames = set()
for line in lines:
    parts = line.split("\t")
    if len(parts) >= 2:
        fname = parts[0].split("#")[0]
        filenames.add(fname)
n_imgs = len(filenames)
print(f"  Images: {n_imgs} | Captions: {n_caps}")

missing = 0
for fname in list(filenames)[:100]:
    if not os.path.exists(f"{img_flickr8k}/{fname}"):
        missing += 1
        if missing <= 3:
            print(f"MISSING: {fname}")
if missing == 0:
    print(f"Image check passed (100 samples)")
else:
    errors.append(f"Arabic: {missing}/100 images missing from flickr8k")
    print(f"{missing}/100 sample images missing!")

# ============ 3. BENGALI ============
print("\n--- BENGALI ---")
import pandas as pd
df = pd.read_csv(f"{base}/bengali/BAN-Cap_captiondata.csv")
filenames_bn = set(df["caption_id"].apply(lambda x: x.split("#")[0]))
print(f"  Images: {len(filenames_bn)} | Captions: {len(df)}")
print(f"  Columns: {list(df.columns)}")

missing = 0
for fname in list(filenames_bn)[:100]:
    if not os.path.exists(f"{img_flickr8k}/{fname}"):
        missing += 1
        if missing <= 3:
            print(f"MISSING: {fname}")
if missing == 0:
    print(f"Image check passed (100 samples)")
else:
    errors.append(f"Bengali: {missing}/100 images missing from flickr8k")
    print(f"{missing}/100 sample images missing!")

# ============ 4. GERMAN ============
print("\n--- GERMAN ---")
total_de = 0
for split in ["train", "val", "test"]:
    with open(f"{base}/german/{split}.de", "r", encoding="utf-8") as f:
        n = len(f.readlines())
    print(f"  {split}: {n} captions")
    total_de += n
print(f"  Total: {total_de}")

# German Multi30k uses numbered images - check flickr30k has enough
flickr30k_imgs = os.listdir(img_flickr30k)
print(f"  Flickr30k has {len(flickr30k_imgs)} images (need >= {total_de} for train)")
if len(flickr30k_imgs) >= 29000:
    print(f"Sufficient images")
else:
    errors.append(f"German: flickr30k might not have enough images")
    print(f"Might not have enough images!")

# ============ 5. ENGLISH ============
print("\n--- ENGLISH ---")
total_en = 0
for split in ["train", "val", "test"]:
    with open(f"{base}/english/{split}.en", "r", encoding="utf-8") as f:
        n = len(f.readlines())
    print(f"  {split}: {n} captions")
    total_en += n
print(f"  Total: {total_en}")
print(f"  Same images as German (Flickr30k)")

# ============ 6. UKRAINIAN ============
print("\n--- UKRAINIAN ---")
total_uk = 0
for fname in ["train.json", "test_2016_flickr.json", "test_2017_flickr.json"]:
    with open(f"{base}/ukrainian/{fname}", "r", encoding="utf-8") as f:
        items = [json.loads(line) for line in f if line.strip()]
    print(f"  {fname}: {len(items)} pairs | keys: {list(items[0].keys())}")
    total_uk += len(items)
print(f"  Total: {total_uk}")
print(f"  Same images as German/English (Flickr30k)")

# ============ 7. VIETNAMESE ============
print("\n--- VIETNAMESE ---")
total_vi_imgs = 0
total_vi_caps = 0
for split in ["train", "dev", "test"]:
    with open(f"{base}/vietnamese/vi_{split}.json", "r", encoding="utf-8") as f:
        data = json.load(f)
    n_imgs = len(data)
    n_caps = sum(len(v["captions"]) for v in data.values())
    print(f"  {split}: {n_imgs} images, {n_caps} captions")
    total_vi_imgs += n_imgs
    total_vi_caps += n_caps

# Check if Vietnamese image files exist
missing = 0
vi_img_files = set(os.listdir(img_viet))
with open(f"{base}/vietnamese/vi_train.json", "r", encoding="utf-8") as f:
    vi_data = json.load(f)
for fname in list(vi_data.keys())[:100]:
    if fname not in vi_img_files:
        missing += 1
        if missing <= 3:
            print(f"  MISSING: {fname}")
if missing == 0:
    print(f"  Image check passed (100 samples)")
else:
    errors.append(f"Vietnamese: {missing}/100 images missing")
    print(f"  {missing}/100 sample images missing!")

print(f"  Total: {total_vi_imgs} images, {total_vi_caps} captions")

# ============ FINAL SUMMARY ============
print("\n" + "=" * 60)
print(" FINAL SUMMARY")
print("=" * 60)
print(f"  {'Language':<12} {'Images':>8} {'Captions':>10} {'Image Source':<15}")
print(f"  {'-'*12} {'-'*8} {'-'*10} {'-'*15}")
print(f"  {'Turkish':<12} {'8,000':>8} {'16,037':>10} {'Flickr8k':<15}")
print(f"  {'Arabic':<12} {str(n_imgs)+' ':>8} {str(n_caps)+' ':>10} {'Flickr8k':<15}")
print(f"  {'Bengali':<12} {str(len(filenames_bn)):>8} {str(len(df)):>10} {'Flickr8k':<15}")
print(f"  {'English':<12} {'~29,000':>8} {str(total_en):>10} {'Flickr30k':<15}")
print(f"  {'German':<12} {'~29,000':>8} {str(total_de):>10} {'Flickr30k':<15}")
print(f"  {'Ukrainian':<12} {'~29,000':>8} {str(total_uk):>10} {'Flickr30k':<15}")
print(f"  {'Vietnamese':<12} {str(total_vi_imgs):>8} {str(total_vi_caps):>10} {'UIT-OpenViIC':<15}")

if errors:
    print(f"\n ERRORS FOUND:")
    for e in errors:
        print(f"  ❌ {e}")
else:
    print(f"\n✅ ALL CHECKS PASSED! Ready for training pipeline.")

🔍 COMPREHENSIVE DATASET VALIDATION

--- TURKISH ---
  Images: 8000 | Captions: 16037
  ✅ Image check passed (100 samples)

--- ARABIC ---
  Images: 8091 | Captions: 24274
  ✅ Image check passed (100 samples)

--- BENGALI ---
  Images: 8091 | Captions: 40455
  Columns: ['caption_id', 'english_caption', 'bengali_caption']
  ✅ Image check passed (100 samples)

--- GERMAN ---
  train: 29000 captions
  val: 1014 captions
  test: 1000 captions
  Total: 31014
  Flickr30k has 31785 images (need >= 31014 for train)
  ✅ Sufficient images

--- ENGLISH ---
  train: 29000 captions
  val: 1014 captions
  test: 1000 captions
  Total: 31014
  ✅ Same images as German (Flickr30k)

--- UKRAINIAN ---
  train.json: 29000 pairs | keys: ['en', 'uk']
  test_2016_flickr.json: 1000 pairs | keys: ['en', 'uk']
  test_2017_flickr.json: 1000 pairs | keys: ['en', 'uk']
  Total: 31000
  ✅ Same images as German/English (Flickr30k)

--- VIETNAMESE ---
  train: 9088 images, 41238 captions
  dev: 2011 images, 10002 capti

In [7]:
import json, os
import pandas as pd

results = []

# Turkish
with open(f"{base}/turkish/tasviret8k_captions.json", "r", encoding="utf-8") as f:
    data = json.load(f)
results.append(("Turkish", len(data["images"]), sum(len(img["sentences"]) for img in data["images"]), "Flickr8k"))

# Arabic
with open(f"{base}/arabic/Flickr8k.arabic.full.txt", "r", encoding="utf-8") as f:
    lines = [l.strip() for l in f if l.strip()]
ar_files = set(l.split("\t")[0].split("#")[0] for l in lines if "\t" in l)
results.append(("Arabic", len(ar_files), len(lines), "Flickr8k"))

# Bengali
df = pd.read_csv(f"{base}/bengali/BAN-Cap_captiondata.csv")
bn_files = set(df["caption_id"].apply(lambda x: x.split("#")[0]))
results.append(("Bengali", len(bn_files), len(df), "Flickr8k"))

# English
en_total = 0
for split in ["train", "val", "test"]:
    with open(f"{base}/english/{split}.en", "r") as f:
        en_total += len(f.readlines())
results.append(("English", "~31K (Flickr30k)", en_total, "Flickr30k"))

# German
de_total = 0
for split in ["train", "val", "test"]:
    with open(f"{base}/german/{split}.de", "r") as f:
        de_total += len(f.readlines())
results.append(("German", "~31K (Flickr30k)", de_total, "Flickr30k"))

# Ukrainian
uk_total = 0
for fname in ["train.json", "test_2016_flickr.json", "test_2017_flickr.json"]:
    with open(f"{base}/ukrainian/{fname}", "r") as f:
        uk_total += sum(1 for line in f if line.strip())
results.append(("Ukrainian", "~31K (Flickr30k)", uk_total, "Flickr30k"))

# Vietnamese
vi_imgs = 0
vi_caps = 0
for split in ["train", "dev", "test"]:
    with open(f"{base}/vietnamese/vi_{split}.json", "r") as f:
        data = json.load(f)
    vi_imgs += len(data)
    vi_caps += sum(len(v["captions"]) for v in data.values())
results.append(("Vietnamese", vi_imgs, vi_caps, "UIT-OpenViIC"))

# Print
print(f"{'Language':<12} {'Images':>14} {'Captions':>10} {'Source':<15}")
print("-" * 55)
total_caps = 0
for lang, imgs, caps, src in results:
    print(f"{lang:<12} {str(imgs):>14} {caps:>10} {src:<15}")
    total_caps += caps
print("-" * 55)
print(f"{'TOTAL':<12} {'':>14} {total_caps:>10}")

Language             Images   Captions Source         
-------------------------------------------------------
Turkish                8000      16037 Flickr8k       
Arabic                 8091      24274 Flickr8k       
Bengali                8091      40455 Flickr8k       
English      ~31K (Flickr30k)      31014 Flickr30k      
German       ~31K (Flickr30k)      31014 Flickr30k      
Ukrainian    ~31K (Flickr30k)      31000 Flickr30k      
Vietnamese            13100      61241 UIT-OpenViIC   
-------------------------------------------------------
TOTAL                           235035


In [ ]:
%%writefile multilingual_dataset.py

"""
Multilingual Dataset Loader for Task 1 — Typology-Agnostic Baseline

Loads all 7 language datasets from Google Drive, unifies them into a common
format, and provides a PyTorch Dataset compatible with the training pipeline
from english_model_lora.py.

Each sample: (image_path, caption, language_code)

Languages:
  1. English   (en) — Multi30k, Flickr30k images
  2. German    (de) — Multi30k, Flickr30k images
  3. Ukrainian (uk) — Multi30k-uk, Flickr30k images
  4. Arabic    (ar) — Flickr8k Arabic, Flickr8k images
  5. Turkish   (tr) — TasvirEt, Flickr8k images
  6. Bengali   (bn) — BAN-Cap, Flickr8k images
  7. Vietnamese(vi) — UIT-OpenViIC, own images

Usage in Colab:
    from multilingual_dataset import load_all_datasets, MultilingualCaptionDataset

    samples = load_all_datasets(base_path="/content/drive/MyDrive/mlp_project/datasets")
    dataset = MultilingualCaptionDataset(samples, tokenizer, max_length=64)
"""

import json
import os
import pandas as pd
from torch.utils.data import Dataset


# ── Language prompt prefixes (used by mT5 to know which language to generate) ─
LANG_PREFIXES = {
    "en": "English: ",
    "de": "German: ",
    "uk": "Ukrainian: ",
    "ar": "Arabic: ",
    "tr": "Turkish: ",
    "bn": "Bengali: ",
    "vi": "Vietnamese: ",
}


def load_english(base_path):
    """Load English Multi30k captions. Returns list of (image_path, caption, 'en')."""
    samples = []
    img_dir = os.path.join(base_path, "images", "flickr30k")

    # Multi30k uses line number = image index
    # We need the image filename list — Multi30k images are named by their Flickr ID
    # The image order file maps line numbers to filenames
    # For Multi30k, images are ordered — line 0 = first image, etc.
    # We'll load the image filenames from the directory sorted, matching Multi30k order

    # Multi30k provides a separate file for image order, but we don't have it
    # Instead, we use the Flickr30k standard: train_images.txt etc.
    # For now, load captions and pair with image index (to be resolved during feature precompute)

    for split in ["train", "val", "test"]:
        caption_file = os.path.join(base_path, "english", f"{split}.en")
        with open(caption_file, "r", encoding="utf-8") as f:
            for line_idx, line in enumerate(f):
                caption = line.strip()
                if caption:
                    # Image path will be resolved later using Multi30k image order
                    samples.append({
                        "image_source": "flickr30k",
                        "image_id": f"multi30k_{split}_{line_idx}",
                        "caption": caption,
                        "lang": "en",
                    })

    print(f"  English: {len(samples)} samples loaded")
    return samples


def load_german(base_path):
    """Load German Multi30k captions."""
    samples = []

    for split in ["train", "val", "test"]:
        caption_file = os.path.join(base_path, "german", f"{split}.de")
        with open(caption_file, "r", encoding="utf-8") as f:
            for line_idx, line in enumerate(f):
                caption = line.strip()
                if caption:
                    samples.append({
                        "image_source": "flickr30k",
                        "image_id": f"multi30k_{split}_{line_idx}",
                        "caption": caption,
                        "lang": "de",
                    })

    print(f"  German: {len(samples)} samples loaded")
    return samples


def load_ukrainian(base_path):
    """Load Ukrainian Multi30k-uk captions (JSONL format)."""
    samples = []

    files = ["train.json", "test_2016_flickr.json", "test_2017_flickr.json"]
    split_names = ["train", "test2016", "test2017"]

    for fname, split_name in zip(files, split_names):
        filepath = os.path.join(base_path, "ukrainian", fname)
        with open(filepath, "r", encoding="utf-8") as f:
            for line_idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                item = json.loads(line)
                uk_caption = item.get("uk", "")
                if uk_caption:
                    samples.append({
                        "image_source": "flickr30k",
                        "image_id": f"multi30k_{split_name}_{line_idx}",
                        "caption": uk_caption,
                        "lang": "uk",
                    })

    print(f"  Ukrainian: {len(samples)} samples loaded")
    return samples


def load_arabic(base_path):
    """Load Arabic Flickr8k captions. Format: filename#N<tab>caption"""
    samples = []

    filepath = os.path.join(base_path, "arabic", "Flickr8k.arabic.full.txt")
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or "\t" not in line:
                continue
            parts = line.split("\t", 1)
            img_id = parts[0].split("#")[0]  # e.g. "1000268201_693b08cb0e.jpg"
            caption = parts[1]
            samples.append({
                "image_source": "flickr8k",
                "image_id": img_id,
                "caption": caption,
                "lang": "ar",
            })

    print(f"  Arabic: {len(samples)} samples loaded")
    return samples


def load_turkish(base_path):
    """Load Turkish TasvirEt captions. JSON with images list."""
    samples = []

    filepath = os.path.join(base_path, "turkish", "tasviret8k_captions.json")
    with open(filepath, "r", encoding="utf-8") as f:
        data = json.load(f)

    for img in data["images"]:
        filename = img["filename"]
        for sent in img["sentences"]:
            caption = sent["raw"]
            samples.append({
                "image_source": "flickr8k",
                "image_id": filename,
                "caption": caption,
                "lang": "tr",
            })

    print(f"  Turkish: {len(samples)} samples loaded")
    return samples


def load_bengali(base_path):
    """Load Bengali BAN-Cap captions. CSV with caption_id, english_caption, bengali_caption."""
    samples = []

    filepath = os.path.join(base_path, "bengali", "BAN-Cap_captiondata.csv")
    df = pd.read_csv(filepath)

    for _, row in df.iterrows():
        img_id = row["caption_id"].split("#")[0]
        caption = row["bengali_caption"]
        if pd.notna(caption) and str(caption).strip():
            samples.append({
                "image_source": "flickr8k",
                "image_id": img_id,
                "caption": str(caption).strip(),
                "lang": "bn",
            })

    print(f"  Bengali: {len(samples)} samples loaded")
    return samples


def load_vietnamese(base_path):
    """Load Vietnamese UIT-OpenViIC captions. JSON with filename -> {captions: [...]}."""
    samples = []

    for split in ["train", "dev", "test"]:
        filepath = os.path.join(base_path, "vietnamese", f"vi_{split}.json")
        with open(filepath, "r", encoding="utf-8") as f:
            data = json.load(f)

        for filename, entry in data.items():
            for caption in entry["captions"]:
                samples.append({
                    "image_source": "vietnamese",
                    "image_id": filename,
                    "caption": caption,
                    "lang": "vi",
                })

    print(f"  Vietnamese: {len(samples)} samples loaded")
    return samples


def load_all_datasets(base_path):
    """
    Load all 7 language datasets and return a unified list of samples.

    Each sample is a dict:
        {
            "image_source": "flickr8k" | "flickr30k" | "vietnamese",
            "image_id": str,   # filename or multi30k index
            "caption": str,
            "lang": str,       # ISO code: en, de, uk, ar, tr, bn, vi
        }
    """
    print("Loading all datasets...")

    all_samples = []
    all_samples.extend(load_english(base_path))
    all_samples.extend(load_german(base_path))
    all_samples.extend(load_ukrainian(base_path))
    all_samples.extend(load_arabic(base_path))
    all_samples.extend(load_turkish(base_path))
    all_samples.extend(load_bengali(base_path))
    all_samples.extend(load_vietnamese(base_path))

    # Summary
    print(f"\n{'='*50}")
    print(f"TOTAL: {len(all_samples)} samples across 7 languages")
    print(f"{'='*50}")

    lang_counts = {}
    for s in all_samples:
        lang_counts[s["lang"]] = lang_counts.get(s["lang"], 0) + 1
    for lang, count in sorted(lang_counts.items()):
        print(f"  {lang}: {count:>6} samples")

    return all_samples


def get_image_path(sample, base_path):
    """
    Resolve the full image path for a sample.

    For flickr8k/vietnamese: image_id is the actual filename.
    For flickr30k (multi30k): needs the image order file (handled separately).
    """
    source = sample["image_source"]
    img_id = sample["image_id"]

    if source == "flickr8k":
        return os.path.join(base_path, "images", "flickr8k", img_id)
    elif source == "vietnamese":
        return os.path.join(base_path, "images", "vietnamese", img_id)
    elif source == "flickr30k":
        # Multi30k images need special handling — see build_multi30k_image_map()
        return None  # resolved later
    return None


def build_multi30k_image_map(base_path):
    """
    Build a mapping from multi30k line indices to Flickr30k image filenames.

    Multi30k images are ordered — we need the image order file.
    This is typically train_images.txt, val_images.txt, test_images.txt
    with one filename per line.

    If these files don't exist, we'll need to download them separately.
    Returns: dict mapping "multi30k_{split}_{line_idx}" -> "filename.jpg"
    """
    img_map = {}
    img_dir = os.path.join(base_path, "images", "flickr30k")

    # Try to load image order files
    for split, order_file in [("train", "train_images.txt"),
                               ("val", "val_images.txt"),
                               ("test", "test_images.txt")]:
        order_path = os.path.join(base_path, "flickr30k_order", order_file)
        if os.path.exists(order_path):
            with open(order_path, "r") as f:
                for line_idx, line in enumerate(f):
                    fname = line.strip()
                    if fname:
                        key = f"multi30k_{split}_{line_idx}"
                        img_map[key] = os.path.join(img_dir, fname)

    if not img_map:
        print("  Multi30k image order files not found!")
        print("  Need to download train_images.txt, val_images.txt, test_images.txt")
        print("  from https://github.com/multi30k/dataset/tree/master/data/task1/image_splits/")

    return img_map


class MultilingualCaptionDataset(Dataset):
    """
    PyTorch Dataset for multilingual captioning.

    Each item returns:
        - image_path (str): full path to the image file
        - caption (str): caption text with language prefix
        - lang (str): language code
    """
    def __init__(self, samples, add_lang_prefix=True):
        self.samples = samples
        self.add_lang_prefix = add_lang_prefix

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        sample = self.samples[idx]
        caption = sample["caption"]
        lang = sample["lang"]

        if self.add_lang_prefix:
            caption = LANG_PREFIXES[lang] + caption

        return {
            "image_source": sample["image_source"],
            "image_id": sample["image_id"],
            "caption": caption,
            "lang": lang,
        }


# ── Quick test ────────────────────────────────────────────────────────────────
if __name__ == "__main__":
    import sys

    base = sys.argv[1] if len(sys.argv) > 1 else "/content/drive/MyDrive/mlp_project/datasets"

    if not os.path.exists(base):
        print(f"Path not found: {base}")
        print("Usage: python multilingual_dataset.py /path/to/datasets")
        sys.exit(1)

    samples = load_all_datasets(base)

    # Show a few samples per language
    print("\nSample captions:")
    shown = set()
    for s in samples:
        if s["lang"] not in shown:
            shown.add(s["lang"])
            prefix = LANG_PREFIXES[s["lang"]]
            print(f"  [{s['lang']}] {prefix}{s['caption'][:80]}...")
        if len(shown) == 7:
            break


Overwriting multilingual_dataset.py


In [9]:
from multilingual_dataset import load_all_datasets
samples = load_all_datasets(base)

Loading all datasets...
  English: 31014 samples loaded
  German: 31014 samples loaded
  Ukrainian: 31000 samples loaded
  Arabic: 24273 samples loaded
  Turkish: 16037 samples loaded
  Bengali: 40455 samples loaded
  Vietnamese: 61241 samples loaded

TOTAL: 235034 samples across 7 languages
  ar:  24273 samples
  bn:  40455 samples
  de:  31014 samples
  en:  31014 samples
  tr:  16037 samples
  uk:  31000 samples
  vi:  61241 samples


In [ ]:
from multilingual_dataset import get_image_path, build_multi30k_image_map

# Build Multi30k image mapping
img_map = build_multi30k_image_map(base)
print(f"Multi30k image map: {len(img_map)} entries")


# Test each language - check if images actually exist
import random
random.seed(42)

errors = []
for lang in ["en", "de", "uk", "ar", "tr", "bn", "vi"]:
    lang_samples = [s for s in samples if s["lang"] == lang]
    test_samples = random.sample(lang_samples, min(20, len(lang_samples)))

    missing = 0
    for s in test_samples:
        if s["image_source"] == "flickr30k":
            path = img_map.get(s["image_id"])
        else:
            path = get_image_path(s, base)

        if path is None or not os.path.exists(path):
            missing += 1
            if missing <= 2:
                print(f" [{lang}] missing: {s['image_id']} → {path}")

    status = "pass" if missing == 0 else f"{missing}/20 missing"
    print(f"  {lang}: {status}")
    if missing > 0:
        errors.append(lang)

if not errors:
    print("\n All image mappings verified! Ready for feature precomputation.")
else:
    print(f"\n Issues found for: {errors}")

Multi30k image map: 31014 entries
  en: ✅
  de: ✅
  uk: ✅
  ar: ✅
  tr: ✅
  bn: ✅
  vi: ✅

✅ All image mappings verified! Ready for feature precomputation.


In [3]:
!nvidia-smi

Sat Mar 21 21:03:23 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla P100-PCIE-16GB           Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P0             27W /  250W |       0MiB /  16384MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [12]:
# 
import gc
import torch
gc.collect()
torch.cuda.empty_cache()

In [ ]:
path = "/kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt"
# path = "/kaggle/working/multilingual_checkpoints/latest_checkpoint.pt"
if os.path.exists(path):
    size = os.path.getsize(path) / 1e9
    print(f" best_checkpoint.pt found: {size:.1f} GB")
else:
    print(" Not found yet")

✅ best_checkpoint.pt found: 3.9 GB


In [15]:

LANG_PREFIXES = {
    "en": "English: ",
    "de": "Deutsch:",
    "ar": "العربية:",
    "vi": "Việt:",
    "tr": "Türk:",
    "bn": "বাংলা:",
    "uk": "Укр:",
}

LANG_BOS = {}
for lang, prefix in LANG_PREFIXES.items():
    ids = mt5_tokenizer(prefix, add_special_tokens=False).input_ids
    LANG_BOS[lang] = ids[0]
    print(f"  {lang}: '{prefix}' → bos={ids[0]}")

  en: 'English: ' → bos=5413
  de: 'Deutsch:' → bos=22606
  ar: 'العربية:' → bos=17112
  vi: 'Việt:' → bos=4674
  tr: 'Türk:' → bos=13374
  bn: 'বাংলা:' → bos=46010
  uk: 'Укр:' → bos=153111


## Training — O1-bos (Forced BOS, No Lang Embedding)

Trains ProjectionMLP + mT5 + LoRA jointly from the English checkpoint.
Language steering via `forced_bos_token_id` only — no lang_embedding.

Architecture:
  BLIP-2 Q-Former features → ProjectionMLP → mT5-base + LoRA
  Encoder input: projected features (B, 32, 768) — no prepended token
  Language identity: native script prefix as forced first decoded token

Training config:
  Epochs: 5 | LR: 1e-4 | Batch: 16 | Balanced sampler
  Checkpoint: best_multilingual_bos.pt

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm

# ── Kaggle paths ──────────────────────────────────────────────────────────────
FEATURE_FILE   = "/kaggle/input/datasets/kavitasriram19/mlp-multilingual-features/multilingual_features.pt"
CHECKPOINT_IN  = "/kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt"
CHECKPOINT_DIR = "/kaggle/working/multilingual_checkpoints/"
DATASET_DIR    = "/kaggle/working/datasets"
CKPT_BOS       = os.path.join(CHECKPOINT_DIR, "best_multilingual_bos.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

# ── Language config ───────────────────────────────────────────────────────────
LANG_TO_ID = {"en": 0, "de": 1, "uk": 2, "ar": 3, "tr": 4, "bn": 5, "vi": 6}
ID_TO_LANG = {v: k for k, v in LANG_TO_ID.items()}
NUM_LANGS  = len(LANG_TO_ID)

# Native script prefixes — unique first tokens verified (none map to 259)
LANG_PREFIXES_BOS = {
    "en": "English: ",
    "de": "Deutsch:",
    "ar": "العربية:",
    "vi": "Việt:",
    "tr": "Türk:",
    "bn": "বাংলা:",
    "uk": "Укр:",
}

# ── 1. Load precomputed features ──────────────────────────────────────────────
print("\n── 1. Load Precomputed Features ─────────────────────────────────────")
assert os.path.exists(FEATURE_FILE), f"Not found: {FEATURE_FILE}"

saved        = torch.load(FEATURE_FILE, weights_only=False)
all_features = saved["features"]        # (52205, 32, 768)
image_ids    = saved["image_ids"]
image_id_to_feat_idx = {iid: i for i, iid in enumerate(image_ids)}
print(f"  {len(image_ids):,} features | shape: {all_features.shape}")
assert not torch.isnan(all_features).any(), "NaN in features"
print("  Feature health check passed")

# ── 2. Load multilingual dataset ──────────────────────────────────────────────
print("\n── 2. Load Multilingual Dataset ──────────────────────────────────────")
from multilingual_dataset import load_all_datasets, build_multi30k_image_map

all_samples = load_all_datasets(DATASET_DIR)
img_map     = build_multi30k_image_map(DATASET_DIR)

resolved_samples = []
skipped = 0
for s in all_samples:
    img_id = s["image_id"]
    if img_id not in image_id_to_feat_idx:
        skipped += 1
        continue
    lang = s["lang"]
    resolved_samples.append({
        "feat_idx": image_id_to_feat_idx[img_id],
        "caption":  LANG_PREFIXES_BOS[lang] + s["caption"],
        "lang":     lang,
        "lang_id":  LANG_TO_ID[lang],
    })

print(f"  Resolved: {len(resolved_samples):,} samples (skipped {skipped})")
for lang, count in sorted(
    {s["lang"]: sum(1 for x in resolved_samples if x["lang"] == lang)
     for s in resolved_samples}.items()
):
    print(f"    {lang}: {count:>6,}")

# ── 3. Load mT5 + LoRA ────────────────────────────────────────────────────────
print("\n── 3. Load mT5-base + LoRA ───────────────────────────────────────────")
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-base").to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"],
)
mt5_model = get_peft_model(mt5_model, lora_config)
mt5_model.print_trainable_parameters()

# ── 4. ProjectionMLP (no lang_embedding) ─────────────────────────────────────
print("\n── 4. ProjectionMLP ──────────────────────────────────────────────────")

class ProjectionMLP(nn.Module):
    def __init__(self, in_dim=768, out_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

projection = ProjectionMLP().to(device)
print(f"  Projection params: {sum(p.numel() for p in projection.parameters()):,}")
print("  No lang_embedding — language steering via forced_bos only")

# ── 5. Compute forced_bos per language ────────────────────────────────────────
print("\n── 5. Forced BOS Tokens ──────────────────────────────────────────────")
LANG_BOS = {}
for lang, prefix in LANG_PREFIXES_BOS.items():
    ids = mt5_tokenizer(prefix, add_special_tokens=False).input_ids
    LANG_BOS[lang] = ids[0]
    print(f"  {lang}: '{prefix}' → bos={ids[0]}")

assert len(set(LANG_BOS.values())) == len(LANG_BOS), \
    "BOS token collision detected"
print("All forced_bos tokens unique")

# ── 6. Load English checkpoint ────────────────────────────────────────────────
print("\n── 6. Load English Checkpoint ────────────────────────────────────────")
assert os.path.exists(CHECKPOINT_IN), f"Not found: {CHECKPOINT_IN}"

ckpt = torch.load(CHECKPOINT_IN, weights_only=True, map_location=device)
projection.load_state_dict(ckpt["projection"])
mt5_model.load_state_dict(ckpt["lora"])
print(f"  Loaded from: {CHECKPOINT_IN}")

start_epoch   = 0
best_val_loss = float("inf")

# ── 7. Image-level train/val split ────────────────────────────────────────────
print("\n── 7. Train/Val Split (image-level, seed=42) ─────────────────────────")
unique_feat_idxs = sorted(set(s["feat_idx"] for s in resolved_samples))
g       = torch.Generator().manual_seed(42)
perm    = torch.randperm(len(unique_feat_idxs), generator=g).tolist()
shuffled = [unique_feat_idxs[i] for i in perm]

split          = int(0.9 * len(shuffled))
train_feat_set = set(shuffled[:split])
val_feat_set   = set(shuffled[split:])

train_indices = [i for i, s in enumerate(resolved_samples)
                 if s["feat_idx"] in train_feat_set]
val_indices   = [i for i, s in enumerate(resolved_samples)
                 if s["feat_idx"] in val_feat_set]

train_imgs = {resolved_samples[i]["feat_idx"] for i in train_indices}
val_imgs   = {resolved_samples[i]["feat_idx"] for i in val_indices}
assert len(train_imgs & val_imgs) == 0, " IMAGE LEAKAGE DETECTED"
print(f"  Train: {len(train_imgs):,} images | Val: {len(val_imgs):,} images ✅")

# ── 8. Dataset and DataLoaders ────────────────────────────────────────────────
print("\n── 8. Dataset and DataLoaders ────────────────────────────────────────")

class MultilingualPrecomputedDataset(Dataset):
    def __init__(self, samples, features, tokenizer, max_length=64):
        self.samples   = samples
        self.features  = features
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s      = self.samples[idx]
        feature = self.features[s["feat_idx"]]
        labels  = self.tokenizer(
            s["caption"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        # No lang_id returned — not needed for bos model
        return feature, labels

full_dataset  = MultilingualPrecomputedDataset(
    resolved_samples, all_features, mt5_tokenizer
)
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset   = torch.utils.data.Subset(full_dataset, val_indices)

train_lang_counts = {}
for idx in train_indices:
    lang = resolved_samples[idx]["lang"]
    train_lang_counts[lang] = train_lang_counts.get(lang, 0) + 1

sample_weights    = [1.0 / train_lang_counts[resolved_samples[i]["lang"]]
                     for i in train_indices]
min_lang_count    = min(train_lang_counts.values())
samples_per_epoch = NUM_LANGS * min_lang_count

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=samples_per_epoch,
    replacement=True,
)
train_loader = DataLoader(
    train_dataset, batch_size=16, sampler=sampler,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f"  ~{min_lang_count:,} samples/lang/epoch | "
      f"Train batches: {len(train_loader)} | Val: {len(val_loader)}")

# ── 9. Caption preview helper ─────────────────────────────────────────────────
preview_samples = {}
for idx in val_dataset.indices:
    s    = resolved_samples[idx]
    lang = s["lang"]
    if lang not in preview_samples:
        preview_samples[lang] = []
    if len(preview_samples[lang]) < 2:
        preview_samples[lang].append(idx)

def preview_captions_bos(epoch_label="untrained"):
    projection.eval()
    mt5_model.eval()
    print(f"\n  ── Caption preview [{epoch_label}] ─────────────────────────")
    for lang in sorted(preview_samples.keys()):
        for idx in preview_samples[lang]:
            s       = resolved_samples[idx]
            feature = all_features[s["feat_idx"]].unsqueeze(0).to(device)
            bos     = LANG_BOS[lang]
            ref     = s["caption"]
            with torch.no_grad():
                enc = projection(feature)    # (1, 32, 768)
                out = mt5_model.generate(
                    inputs_embeds=enc,
                    forced_bos_token_id=bos,
                    max_new_tokens=50,
                    num_beams=4,
                )
            pred = mt5_tokenizer.decode(out[0], skip_special_tokens=True)
            print(f"  [{lang}] PRED: {pred}")
            print(f"          REF:  {ref[:100]}")
    print()

# ── 10. Training and validation loops ─────────────────────────────────────────
trainable_params = (
    list(projection.parameters()) +
    [p for p in mt5_model.parameters() if p.requires_grad]
)

def train_epoch_bos(loader, optimizer):
    projection.train()
    mt5_model.train()
    total_loss = total_gnorm = 0.0

    for features, labels in tqdm(loader, desc="Train", leave=False):
        features = features.to(device)
        labels   = labels.to(device)

        encoder_input   = projection(features)    # (B, 32, 768)
        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(
            inputs_embeds=encoder_input,
            labels=labels_for_loss,
        ).loss

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError("NaN/Inf loss detected")

        optimizer.zero_grad()
        loss.backward()
        gnorm = torch.nn.utils.clip_grad_norm_(
            trainable_params, max_norm=1.0
        )
        optimizer.step()
        total_loss  += loss.item()
        total_gnorm += gnorm.item()

    return total_loss / len(loader), total_gnorm / len(loader)


@torch.no_grad()
def val_epoch_bos(loader):
    projection.eval()
    mt5_model.eval()
    total_loss = 0.0

    for features, labels in tqdm(loader, desc="Val", leave=False):
        features = features.to(device)
        labels   = labels.to(device)

        encoder_input   = projection(features)
        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(
            inputs_embeds=encoder_input,
            labels=labels_for_loss,
        ).loss
        total_loss += loss.item()

    return total_loss / len(loader)

# ── 11. Run training ──────────────────────────────────────────────────────────
EPOCHS = 5
LR     = 1e-4

optimizer = torch.optim.AdamW(
    trainable_params, lr=LR, weight_decay=1e-2
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

preview_captions_bos(epoch_label=f"epoch 0 — before training")

for epoch in range(start_epoch, EPOCHS):
    train_loss, gnorm = train_epoch_bos(train_loader, optimizer)
    val_loss          = val_epoch_bos(val_loader)
    scheduler.step()

    print(
        f"Epoch {epoch+1:2d}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"GradNorm: {gnorm:.3f} | lr: {scheduler.get_last_lr()[0]:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "projection": projection.state_dict(),
                "lora":       mt5_model.state_dict(),
                "epoch":      epoch + 1,
                "val_loss":   val_loss,
            },
            CKPT_BOS,
        )
        print(f"  Best model saved (val_loss={val_loss:.4f})")

    if (epoch + 1) % 2 == 0 or epoch == EPOCHS - 1:
        preview_captions_bos(epoch_label=f"epoch {epoch+1}")

print(f"\n O1-bos training complete")
print(f"   Best val loss: {best_val_loss:.4f}")
print(f"   Checkpoint: {CKPT_BOS}")

## Training — O1-lang (Lang Embedding)

Trains ProjectionMLP + LangEmbedding + mT5 + LoRA jointly.
Language steering via a learned 768-dim embedding prepended to
visual features at encoder position 0.

Architecture:
  BLIP-2 Q-Former features → ProjectionMLP
  → [lang_embedding | projected_features] → mT5-base + LoRA
  Encoder input: (B, 33, 768) — position 0 is the language token

Key difference from O1-bos:
  - lang_embedding persists throughout generation via cross-attention
  - forced_bos not used — generation is free after the start token
  - Decoder sees language identity at every step, not just step 1

Training config:
  Epochs: 10 | LR: 1e-4 | Batch: 16 | Balanced sampler
  Checkpoint: best_multilingual.pt

In [ ]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm

# ── Kaggle paths ──────────────────────────────────────────────────────────────
FEATURE_FILE   = "/kaggle/input/datasets/kavitasriram19/mlp-multilingual-features/multilingual_features.pt"
CHECKPOINT_IN  = "/kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt"
CHECKPOINT_DIR = "/kaggle/working/multilingual_checkpoints/"
DATASET_DIR    = "/kaggle/working/datasets"
CKPT_LANG      = os.path.join(CHECKPOINT_DIR, "best_multilingual.pt")

os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

# ── Language config ───────────────────────────────────────────────────────────
LANG_TO_ID = {"en": 0, "de": 1, "uk": 2, "ar": 3, "tr": 4, "bn": 5, "vi": 6}
ID_TO_LANG = {v: k for k, v in LANG_TO_ID.items()}
NUM_LANGS  = len(LANG_TO_ID)

LANG_PREFIXES_LANG = {
    "en": "English: ",
    "de": "Deutsch:",
    "ar": "العربية:",
    "vi": "Việt:",
    "tr": "Türk:",
    "bn": "বাংলা:",
    "uk": "Укр:",
}

# ── 1. Load precomputed features ──────────────────────────────────────────────
print("\n── 1. Load Precomputed Features ─────────────────────────────────────")
assert os.path.exists(FEATURE_FILE), f"Not found: {FEATURE_FILE}"

saved        = torch.load(FEATURE_FILE, weights_only=False)
all_features = saved["features"]
image_ids    = saved["image_ids"]
image_id_to_feat_idx = {iid: i for i, iid in enumerate(image_ids)}
print(f"  {len(image_ids):,} features | shape: {all_features.shape}")
assert not torch.isnan(all_features).any(), "NaN in features"
print("  Feature health check passed")

# ── 2. Load multilingual dataset ──────────────────────────────────────────────
print("\n── 2. Load Multilingual Dataset ──────────────────────────────────────")
from multilingual_dataset import load_all_datasets, build_multi30k_image_map

all_samples = load_all_datasets(DATASET_DIR)

resolved_samples = []
skipped = 0
for s in all_samples:
    img_id = s["image_id"]
    if img_id not in image_id_to_feat_idx:
        skipped += 1
        continue
    lang = s["lang"]
    resolved_samples.append({
        "feat_idx": image_id_to_feat_idx[img_id],
        "caption":  LANG_PREFIXES_LANG[lang] + s["caption"],
        "lang":     lang,
        "lang_id":  LANG_TO_ID[lang],
    })

print(f"  Resolved: {len(resolved_samples):,} samples (skipped {skipped})")
for lang, count in sorted(
    {s["lang"]: sum(1 for x in resolved_samples if x["lang"] == lang)
     for s in resolved_samples}.items()
):
    print(f"    {lang}: {count:>6,}")

# ── 3. Load mT5 + LoRA ────────────────────────────────────────────────────────
print("\n── 3. Load mT5-base + LoRA ───────────────────────────────────────────")
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
mt5_model     = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-base").to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"],
)
mt5_model = get_peft_model(mt5_model, lora_config)
mt5_model.print_trainable_parameters()

# ── 4. ProjectionMLP + LangEmbedding ─────────────────────────────────────────
print("\n── 4. ProjectionMLP + LangEmbedding ─────────────────────────────────")

class ProjectionMLP(nn.Module):
    def __init__(self, in_dim=768, out_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

projection     = ProjectionMLP().to(device)
lang_embedding = nn.Embedding(NUM_LANGS, 768).to(device)

print(f"  Projection params:    {sum(p.numel() for p in projection.parameters()):,}")
print(f"  LangEmbedding params: {sum(p.numel() for p in lang_embedding.parameters()):,}")
print("  lang_embedding prepended at encoder position 0 — "
      "cross-attended throughout generation")

# ── 5. Load English checkpoint ────────────────────────────────────────────────
print("\n── 5. Load English Checkpoint ────────────────────────────────────────")
assert os.path.exists(CHECKPOINT_IN), f"Not found: {CHECKPOINT_IN}"

ckpt = torch.load(CHECKPOINT_IN, weights_only=True, map_location=device)
projection.load_state_dict(ckpt["projection"])
mt5_model.load_state_dict(ckpt["lora"])
# lang_embedding is new — not in English checkpoint, initialised randomly
print(f"  Loaded projection + LoRA from: {CHECKPOINT_IN}")
print("  lang_embedding: randomly initialised (trained from scratch)")

start_epoch   = 0
best_val_loss = float("inf")

# ── 6. Image-level train/val split ────────────────────────────────────────────
print("\n── 6. Train/Val Split (image-level, seed=42) ─────────────────────────")
unique_feat_idxs = sorted(set(s["feat_idx"] for s in resolved_samples))
g       = torch.Generator().manual_seed(42)
perm    = torch.randperm(len(unique_feat_idxs), generator=g).tolist()
shuffled = [unique_feat_idxs[i] for i in perm]

split          = int(0.9 * len(shuffled))
train_feat_set = set(shuffled[:split])
val_feat_set   = set(shuffled[split:])

train_indices = [i for i, s in enumerate(resolved_samples)
                 if s["feat_idx"] in train_feat_set]
val_indices   = [i for i, s in enumerate(resolved_samples)
                 if s["feat_idx"] in val_feat_set]

train_imgs = {resolved_samples[i]["feat_idx"] for i in train_indices}
val_imgs   = {resolved_samples[i]["feat_idx"] for i in val_indices}
assert len(train_imgs & val_imgs) == 0, "IMAGE LEAKAGE DETECTED"
print(f"  Train: {len(train_imgs):,} images | Val: {len(val_imgs):,} images ✅")

# ── 7. Dataset and DataLoaders ────────────────────────────────────────────────
print("\n── 7. Dataset and DataLoaders ────────────────────────────────────────")

class MultilingualPrecomputedDatasetLang(Dataset):
    def __init__(self, samples, features, tokenizer, max_length=64):
        self.samples    = samples
        self.features   = features
        self.tokenizer  = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s       = self.samples[idx]
        feature = self.features[s["feat_idx"]]
        labels  = self.tokenizer(
            s["caption"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        # lang_id returned — needed for lang_embedding lookup
        return feature, labels, s["lang_id"]

full_dataset  = MultilingualPrecomputedDatasetLang(
    resolved_samples, all_features, mt5_tokenizer
)
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset   = torch.utils.data.Subset(full_dataset, val_indices)

train_lang_counts = {}
for idx in train_indices:
    lang = resolved_samples[idx]["lang"]
    train_lang_counts[lang] = train_lang_counts.get(lang, 0) + 1

sample_weights    = [1.0 / train_lang_counts[resolved_samples[i]["lang"]]
                     for i in train_indices]
min_lang_count    = min(train_lang_counts.values())
samples_per_epoch = NUM_LANGS * min_lang_count

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=samples_per_epoch,
    replacement=True,
)
train_loader = DataLoader(
    train_dataset, batch_size=16, sampler=sampler,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f"  ~{min_lang_count:,} samples/lang/epoch | "
      f"Train batches: {len(train_loader)} | Val: {len(val_loader)}")

# ── 8. Forward pass helper ────────────────────────────────────────────────────
def encode_with_lang(features, lang_ids):
    """
    Prepend lang_embedding to projected visual features.
    Returns (B, 33, 768) — position 0 is the language identity token.
    """
    projected = projection(features)                    # (B, 32, 768)
    lang_emb  = lang_embedding(lang_ids).unsqueeze(1)   # (B, 1, 768)
    return torch.cat([lang_emb, projected], dim=1)      # (B, 33, 768)

# ── 9. Caption preview helper ─────────────────────────────────────────────────
preview_samples = {}
for idx in val_dataset.indices:
    s    = resolved_samples[idx]
    lang = s["lang"]
    if lang not in preview_samples:
        preview_samples[lang] = []
    if len(preview_samples[lang]) < 2:
        preview_samples[lang].append(idx)

def preview_captions_lang(epoch_label="untrained"):
    projection.eval()
    mt5_model.eval()
    lang_embedding.eval()
    print(f"\n  ── Caption preview [{epoch_label}] ─────────────────────────")
    for lang in sorted(preview_samples.keys()):
        for idx in preview_samples[lang]:
            s       = resolved_samples[idx]
            feature = all_features[s["feat_idx"]].unsqueeze(0).to(device)
            lid     = torch.tensor([s["lang_id"]], device=device)
            ref     = s["caption"]
            with torch.no_grad():
                enc = encode_with_lang(feature, lid)
                out = mt5_model.generate(
                    inputs_embeds=enc,
                    max_new_tokens=50,
                    num_beams=4,
                )
            pred = mt5_tokenizer.decode(out[0], skip_special_tokens=True)
            print(f"  [{lang}] PRED: {pred}")
            print(f"          REF:  {ref[:100]}")
    print()

# ── 10. Training and validation loops ─────────────────────────────────────────
def train_epoch_lang(loader, optimizer):
    projection.train()
    mt5_model.train()
    lang_embedding.train()
    total_loss = total_gnorm = 0.0

    for features, labels, lang_ids in tqdm(loader, desc="Train", leave=False):
        features = features.to(device)
        labels   = labels.to(device)
        lang_ids = lang_ids.to(device)

        encoder_input   = encode_with_lang(features, lang_ids)
        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(
            inputs_embeds=encoder_input,
            labels=labels_for_loss,
        ).loss

        if torch.isnan(loss) or torch.isinf(loss):
            raise RuntimeError("NaN/Inf loss detected")

        optimizer.zero_grad()
        loss.backward()
        all_params = (
            list(projection.parameters()) +
            list(lang_embedding.parameters()) +
            [p for p in mt5_model.parameters() if p.requires_grad]
        )
        gnorm = torch.nn.utils.clip_grad_norm_(all_params, max_norm=1.0)
        optimizer.step()
        total_loss  += loss.item()
        total_gnorm += gnorm.item()

    return total_loss / len(loader), total_gnorm / len(loader)


@torch.no_grad()
def val_epoch_lang(loader):
    projection.eval()
    mt5_model.eval()
    lang_embedding.eval()
    total_loss = 0.0

    for features, labels, lang_ids in tqdm(loader, desc="Val", leave=False):
        features = features.to(device)
        labels   = labels.to(device)
        lang_ids = lang_ids.to(device)

        encoder_input   = encode_with_lang(features, lang_ids)
        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(
            inputs_embeds=encoder_input,
            labels=labels_for_loss,
        ).loss
        total_loss += loss.item()

    return total_loss / len(loader)

# ── 11. Run training ──────────────────────────────────────────────────────────
EPOCHS = 10
LR     = 1e-4

all_params = (
    list(projection.parameters()) +
    list(lang_embedding.parameters()) +
    [p for p in mt5_model.parameters() if p.requires_grad]
)
optimizer = torch.optim.AdamW(all_params, lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

preview_captions_lang(epoch_label="epoch 0 — before training")

for epoch in range(start_epoch, EPOCHS):
    train_loss, gnorm = train_epoch_lang(train_loader, optimizer)
    val_loss          = val_epoch_lang(val_loader)
    scheduler.step()

    print(
        f"Epoch {epoch+1:2d}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"GradNorm: {gnorm:.3f} | lr: {scheduler.get_last_lr()[0]:.2e}"
    )

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "projection":     projection.state_dict(),
                "lora":           mt5_model.state_dict(),
                "lang_embedding": lang_embedding.state_dict(),
                "epoch":          epoch + 1,
                "val_loss":       val_loss,
            },
            CKPT_LANG,
        )
        print(f"  Best model saved (val_loss={val_loss:.4f})")

    if (epoch + 1) % 2 == 0 or epoch == EPOCHS - 1:
        preview_captions_lang(epoch_label=f"epoch {epoch+1}")

print(f"\nO1-lang training complete")
print(f"   Best val loss: {best_val_loss:.4f}")
print(f"   Checkpoint: {CKPT_LANG}")

In [ ]:
"""
Multilingual Training — Typology-Agnostic Baseline (O1) — CORRECTED

Key fix: A language embedding is prepended to the visual features so the model
knows which language to generate in. Without this, the model sees the same image
features for all 7 languages and has no way to decide the output language.

Input to mT5 encoder: [lang_embedding, proj_feat_1, ..., proj_feat_32] = (B, 33, 768)
Labels: "Arabic: ..." / "Bengali: ..." etc. (prefix kept in labels to reinforce)

Also includes WeightedRandomSampler for balanced language representation.

Architecture:
  BLIP-2 Q-Former features  -> precomputed, frozen
  ProjectionMLP 768->768    -> from English checkpoint, fine-tuned
  LangEmbedding 7->768      -> NEW, trained from scratch
  mT5-base + LoRA           -> from English checkpoint, fine-tuned
"""

import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm

# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE_PATH = "/kaggle/working/datasets"
PROJECT_PATH = "/kaggle/working"
FEATURE_FILE = "/kaggle/input/datasets/kavitasriram19/mlp-multilingual-features/multilingual_features.pt"
CHECKPOINT_IN = "/kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt"
CHECKPOINT_DIR = "/kaggle/working/multilingual_checkpoints/"

# os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# ── Language config ───────────────────────────────────────────────────────────
LANG_TO_ID = {"en": 0, "de": 1, "uk": 2, "ar": 3, "tr": 4, "bn": 5, "vi": 6}
ID_TO_LANG = {v: k for k, v in LANG_TO_ID.items()}
NUM_LANGS = len(LANG_TO_ID)

# LANG_PREFIXES = {
#     "en": "English: ",
#     "de": "German: ",
#     "uk": "Ukrainian: ",
#     "ar": "Arabic: ",
#     "tr": "Turkish: ",
#     "bn": "Bengali: ",
#     "vi": "Vietnamese: ",
# }

LANG_PREFIXES = {
    "en": "English: ",
    "de": "Deutsch:",
    "ar": "العربية:",
    "vi": "Việt:",
    "tr": "Türk:",
    "bn": "বাংলা:",
    "uk": "Укр:",
}

# ── 1. Load precomputed features ──────────────────────────────────────────────
print("\n── 1. Load Precomputed Features " + "─" * 38)
assert os.path.exists(FEATURE_FILE), f"Not found: {FEATURE_FILE}"

saved = torch.load(FEATURE_FILE, weights_only=False)
all_features = saved["features"]       # (52205, 32, 768)
image_ids = saved["image_ids"]
image_id_to_feat_idx = {iid: i for i, iid in enumerate(image_ids)}
print(f"  {len(image_ids)} image features, shape: {all_features.shape}")
assert not torch.isnan(all_features).any(), "NaN in features!"
print("  Feature health check passed")

# ── 2. Load multilingual dataset ──────────────────────────────────────────────
print("\n── 2. Load Multilingual Dataset " + "─" * 38)
from multilingual_dataset import load_all_datasets, build_multi30k_image_map

all_samples = load_all_datasets(BASE_PATH)
img_map = build_multi30k_image_map(BASE_PATH)

# Resolve samples: map each to a feature index, add lang_id
resolved_samples = []
skipped = 0

for s in all_samples:
    img_id = s["image_id"]
    if img_id not in image_id_to_feat_idx:
        skipped += 1
        continue

    lang = s["lang"]
    resolved_samples.append({
        "feat_idx": image_id_to_feat_idx[img_id],
        "caption": LANG_PREFIXES[lang] + s["caption"],
        "lang": lang,
        "lang_id": LANG_TO_ID[lang],
    })

print(f"  Resolved: {len(resolved_samples)} samples (skipped {skipped})")

lang_counts = {}
for s in resolved_samples:
    lang_counts[s["lang"]] = lang_counts.get(s["lang"], 0) + 1
for lang, count in sorted(lang_counts.items()):
    print(f"    {lang}: {count:>6}")

# ── 3. Load mT5 + LoRA ───────────────────────────────────────────────────────
print("\n── 3. Load mT5-base + LoRA " + "─" * 43)
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
mt5_model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-base").to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"],
)
mt5_model = get_peft_model(mt5_model, lora_config)
mt5_model.print_trainable_parameters()

# # ── 4. Projection MLP + Language Embedding ────────────────────────────────────
# print("\n── 4. Projection MLP + Language Embedding " + "─" * 28)

# class ProjectionMLP(nn.Module):
#     def __init__(self, in_dim=768, out_dim=768):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(in_dim, out_dim),
#             nn.GELU(),
#             nn.LayerNorm(out_dim),
#             nn.Linear(out_dim, out_dim),
#         )
#     def forward(self, x):
#         return self.net(x)

# projection = ProjectionMLP().to(device)

# # NEW: Language embedding — maps lang_id (0-6) to a 768-dim vector
# # This is prepended to visual features so the model knows which language to use
# lang_embedding = nn.Embedding(NUM_LANGS, 768).to(device)

# proj_params = sum(p.numel() for p in projection.parameters())
# lang_params = sum(p.numel() for p in lang_embedding.parameters())
# print(f"  Projection params: {proj_params:,}")
# print(f"  Lang embedding params: {lang_params:,} (7 languages x 768 dim)")

# ── 4. Projection MLP ─────────────────────────────────────────────────────────
print("\n── 4. Projection MLP " + "─" * 49)

class ProjectionMLP(nn.Module):
    def __init__(self, in_dim=768, out_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

projection = ProjectionMLP().to(device)
proj_params = sum(p.numel() for p in projection.parameters())
print(f"  Projection params: {proj_params:,}")
# No lang_embedding — language steering via forced_bos at inference only

# ── 5. Load English checkpoint ────────────────────────────────────────────────
print("\n── 5. Load English Checkpoint " + "─" * 40)
assert os.path.exists(CHECKPOINT_IN), f"Not found: {CHECKPOINT_IN}"

ckpt = torch.load(CHECKPOINT_IN, weights_only=True, map_location=device)
projection.load_state_dict(ckpt["projection"])
mt5_model.load_state_dict(ckpt["lora"])
# lang_embedding is NEW — not in checkpoint, initialized randomly. This is fine.
print(f"  Loaded projection + LoRA from {CHECKPOINT_IN}")
# print(f"  lang_embedding initialized randomly (new parameter)")

# ── 6. Check for resume ──────────────────────────────────────────────────────
# print("\n── 6. Check for Resume " + "─" * 46)
start_epoch = 0
best_val_loss = float("inf")

# resume_path = os.path.join(CHECKPOINT_DIR, "latest_checkpoint.pt")
# if os.path.exists(resume_path):
#     print(f"  Found resume checkpoint...")
#     resume_ckpt = torch.load(resume_path, weights_only=False, map_location=device)
#     projection.load_state_dict(resume_ckpt["projection"])
#     mt5_model.load_state_dict(resume_ckpt["lora"])
#     lang_embedding.load_state_dict(resume_ckpt["lang_embedding"])
#     start_epoch = resume_ckpt["epoch"]
#     best_val_loss = resume_ckpt.get("best_val_loss", float("inf"))
#     print(f"  Resuming from epoch {start_epoch}, best_val_loss={best_val_loss:.4f}")
# else:
#     print(f"  Starting fresh from English checkpoint + random lang_embedding")


# Precompute forced_bos token per language — first token of prefix
LANG_BOS = {}
for lang, prefix in LANG_PREFIXES.items():
    ids = mt5_tokenizer(prefix, add_special_tokens=False).input_ids
    LANG_BOS[lang] = ids[0]
    print(f"  {lang}: '{prefix}' → bos={ids[0]}")

# Verify no collisions
assert len(set(LANG_BOS.values())) == len(LANG_BOS), "BOS TOKEN COLLISION"
print("All forced_bos tokens unique")

# ── 7. Dataset & Balanced DataLoader ──────────────────────────────────────────
print("\n── 7. Dataset & Balanced DataLoader " + "─" * 34)

class MultilingualPrecomputedDataset(Dataset):
    def __init__(self, samples, features, tokenizer, max_length=64):
        self.samples = samples
        self.features = features
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        feature = self.features[s["feat_idx"]]  # (32, 768)
        # lang_id = s["lang_id"]

        labels = self.tokenizer(
            s["caption"],
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)

        # return feature, labels, lang_id
        return feature, labels

full_dataset = MultilingualPrecomputedDataset(
    resolved_samples, all_features, mt5_tokenizer
)

# 90/10 split
# train_size = int(0.9 * len(full_dataset))
# val_size = len(full_dataset) - train_size
# train_dataset, val_dataset = torch.utils.data.random_split(
#     full_dataset, [train_size, val_size],
#     generator=torch.Generator().manual_seed(42),
# )

# ------------------------------------------------------------------
# IMAGE-LEVEL SPLIT (NO LEAKAGE)
# ------------------------------------------------------------------

# 1. Get all unique image ids (feat_idx = image id)
all_feat_idxs = [s["feat_idx"] for s in resolved_samples]
unique_feat_idxs = sorted(set(all_feat_idxs))

# 2. Shuffle them
g = torch.Generator().manual_seed(42)
perm = torch.randperm(len(unique_feat_idxs), generator=g).tolist()
shuffled_feat_idxs = [unique_feat_idxs[i] for i in perm]

# 3. Split images (NOT samples)
split = int(0.9 * len(shuffled_feat_idxs))
train_feat_set = set(shuffled_feat_idxs[:split])
val_feat_set   = set(shuffled_feat_idxs[split:])

# 4. Assign samples based on image
train_indices = [
    i for i, s in enumerate(resolved_samples)
    if s["feat_idx"] in train_feat_set
]

val_indices = [
    i for i, s in enumerate(resolved_samples)
    if s["feat_idx"] in val_feat_set
]

# 5. Create datasets
train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset   = torch.utils.data.Subset(full_dataset, val_indices)

# 6. Sanity check (VERY IMPORTANT)
train_imgs = {resolved_samples[i]["feat_idx"] for i in train_indices}
val_imgs   = {resolved_samples[i]["feat_idx"] for i in val_indices}

assert len(train_imgs & val_imgs) == 0, "IMAGE LEAKAGE DETECTED"

print(f"Train images: {len(train_imgs)}, Val images: {len(val_imgs)}")
print(f"Train samples: {len(train_indices)}, Val samples: {len(val_indices)}")
print("No image overlap between train and val")

# Balanced sampler
train_indices = train_dataset.indices
train_lang_counts = {}
for idx in train_indices:
    lang = resolved_samples[idx]["lang"]
    train_lang_counts[lang] = train_lang_counts.get(lang, 0) + 1

print("  Train language counts:")
for lang, count in sorted(train_lang_counts.items()):
    print(f"    {lang}: {count:>6}")

sample_weights = []
for idx in train_indices:
    lang = resolved_samples[idx]["lang"]
    sample_weights.append(1.0 / train_lang_counts[lang])

min_lang_count = min(train_lang_counts.values())
samples_per_epoch = NUM_LANGS * min_lang_count
print(f"\n  Balanced: ~{min_lang_count} per language per epoch")
print(f"  Samples per epoch: {samples_per_epoch}")

sampler = WeightedRandomSampler(
    weights=sample_weights,
    num_samples=samples_per_epoch,
    replacement=True,
)

train_loader = DataLoader(
    train_dataset, batch_size=16, sampler=sampler,
    num_workers=2, pin_memory=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False,
    num_workers=2, pin_memory=True,
)
print(f"  Train batches/epoch: {len(train_loader)} | Val: {len(val_loader)}")

# ── 8. Forward pass helper ────────────────────────────────────────────────────
# This is where the language embedding is prepended to visual features.

# def encode_with_lang(features, lang_ids):
#     """
#     features: (B, 32, 768) — projected visual features
#     lang_ids: (B,) — language IDs (0-6)
#     returns:  (B, 33, 768) — [lang_emb, feat_1, ..., feat_32]
#     """
#     projected = projection(features)                    # (B, 32, 768)
#     lang_emb = lang_embedding(lang_ids).unsqueeze(1)    # (B, 1, 768)
#     return torch.cat([lang_emb, projected], dim=1)      # (B, 33, 768)

# ── 9. Caption preview ────────────────────────────────────────────────────────
preview_samples = {}
for idx in val_dataset.indices:
    s = resolved_samples[idx]
    lang = s["lang"]
    if lang not in preview_samples:
        preview_samples[lang] = []
    if len(preview_samples[lang]) < 2:
        preview_samples[lang].append(idx)

# import gc
# gc.collect()
# torch.cuda.empty_cache()

def preview_captions(epoch_label="untrained"):
    projection.eval()
    mt5_model.eval()
    # lang_embedding.eval()
    print(f"\n  ── Caption preview [{epoch_label}] " + "─" * 28)
    for lang in sorted(preview_samples.keys()):
        for idx in preview_samples[lang]:
            s = resolved_samples[idx]
            feature = all_features[s["feat_idx"]].unsqueeze(0).to(device)
            # lid = torch.tensor([s["lang_id"]], device=device)
            ref = s["caption"]
            bos     = LANG_BOS[lang]
            with torch.no_grad():
                # encoder_input = encode_with_lang(feature, lid)  # (1, 33, 768)
                encoder_input = projection(feature) # (1, 32, 768)
                out = mt5_model.generate(
                    inputs_embeds=encoder_input,
                    max_new_tokens=50,
                    num_beams=4,
                    forced_bos_token_id=bos,
                )
            pred = mt5_tokenizer.decode(out[0], skip_special_tokens=True)
            print(f"  [{lang}] PRED: {pred}")
            print(f"       REF:  {ref[:100]}")
    print()

# ── 10. Training loop ─────────────────────────────────────────────────────────
print("\n── 10. Training " + "─" * 54)
trainable_params = (
            list(projection.parameters()) +
            [p for p in mt5_model.parameters() if p.requires_grad]
        )
def train_epoch(loader, optimizer):
    projection.train()
    mt5_model.train()
    # lang_embedding.train()
    total_loss = 0.0
    total_gnorm = 0.0

    # for batch_i, (features, labels, lang_ids) in enumerate(tqdm(loader, desc="Train", leave=False)):
    for batch_i, (features, labels) in enumerate(tqdm(loader, desc="Train", leave=False)):
        features = features.to(device)
        labels = labels.to(device)
        # lang_ids = lang_ids.to(device)

        # encoder_input = encode_with_lang(features, lang_ids)  # (B, 33, 768)
        encoder_input = projection(features)
        
        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(inputs_embeds=encoder_input, labels=labels_for_loss).loss

        if torch.isnan(loss) or torch.isinf(loss):
            print(f"\n  NaN/Inf loss at batch {batch_i}")
            raise RuntimeError("NaN/Inf loss")

        optimizer.zero_grad()
        loss.backward()

        # all_params = (
        #     list(projection.parameters())
        #     + list(mt5_model.parameters())
        #     + list(lang_embedding.parameters())
        # )
        
        gnorm = torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=1.0)
        total_gnorm += gnorm.item()
        optimizer.step()
        total_loss += loss.item()

    return total_loss / len(loader), total_gnorm / len(loader)


@torch.no_grad()
def val_epoch(loader):
    projection.eval()
    mt5_model.eval()
    # lang_embedding.eval()
    total_loss = 0.0

    # for features, labels, lang_ids in tqdm(loader, desc="Val", leave=False):
    for features, labels in tqdm(loader, desc="Val", leave=False):
        features = features.to(device)
        labels = labels.to(device)
        # lang_ids = lang_ids.to(device)

        # encoder_input = encode_with_lang(features, lang_ids)
        encoder_input = projection(features)

        labels_for_loss = labels.clone()
        labels_for_loss[labels_for_loss == mt5_tokenizer.pad_token_id] = -100

        loss = mt5_model(inputs_embeds=encoder_input, labels=labels_for_loss).loss
        total_loss += loss.item()

    return total_loss / len(loader)


# ── Training config ───────────────────────────────────────────────────────────
EPOCHS = 5
LR = 1e-4

# all_params = (
#     list(projection.parameters())
#     + list(mt5_model.parameters())
#     + list(lang_embedding.parameters())
# )
# optimizer = torch.optim.AdamW(all_params, lr=LR, weight_decay=1e-2)

# trainable_params = (
#     list(projection.parameters()) +
#     [p for p in mt5_model.parameters() if p.requires_grad]
# )
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

for _ in range(start_epoch):
    scheduler.step()

# Baseline preview
preview_captions(epoch_label=f"epoch {start_epoch} — before training")
for epoch in range(start_epoch, EPOCHS):
    train_loss, gnorm = train_epoch(train_loader, optimizer)
    val_loss = val_epoch(val_loader)
    scheduler.step()
    lr_now = scheduler.get_last_lr()[0]

    print(
        f"Epoch {epoch + 1:2d}/{EPOCHS} | "
        f"Train: {train_loss:.4f} | Val: {val_loss:.4f} | "
        f"GradNorm: {gnorm:.3f} | lr: {lr_now:.2e}"
    )

    # Save latest (for resume)
    # torch.save(
    #     {
    #         "projection": projection.state_dict(),
    #         "lora": mt5_model.state_dict(),
    #         # "lang_embedding": lang_embedding.state_dict(),
    #         "epoch": epoch + 1,
    #         "train_loss": train_loss,
    #         "val_loss": val_loss,
    #         "best_val_loss": best_val_loss,
    #     },
    #     os.path.join(CHECKPOINT_DIR, "latest_checkpoint_bos.pt"),
    # )

    # Save best
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(
            {
                "projection": projection.state_dict(),
                "lora": mt5_model.state_dict(),
                # "lang_embedding": lang_embedding.state_dict(),
                "epoch": epoch + 1,
                "val_loss": val_loss,
            },
            os.path.join(CHECKPOINT_DIR, "best_multilingual_bos.pt"),
        )
        print(f"  Saved best model (val_loss={val_loss:.4f})")

    # Preview every 2 epochs + final
    if (epoch + 1) % 2 == 0 or epoch == 1:
        preview_captions(epoch_label=f"epoch {epoch + 1}")

print("\nMultilingual training complete!")
print(f"  Best val loss: {best_val_loss:.4f}")
print(f"  Checkpoints: {CHECKPOINT_DIR}")

Device: cuda
GPU: Tesla P100-PCIE-16GB | VRAM: 17.1 GB

── 1. Load Precomputed Features ──────────────────────────────────────
  52205 image features, shape: torch.Size([52205, 32, 768])
  Feature health check passed

── 2. Load Multilingual Dataset ──────────────────────────────────────
Loading all datasets...
  English: 31014 samples loaded
  German: 31014 samples loaded
  Ukrainian: 31000 samples loaded
  Arabic: 24273 samples loaded
  Turkish: 16037 samples loaded
  Bengali: 40455 samples loaded
  Vietnamese: 61241 samples loaded

TOTAL: 235034 samples across 7 languages
  ar:  24273 samples
  bn:  40455 samples
  de:  31014 samples
  en:  31014 samples
  tr:  16037 samples
  uk:  31000 samples
  vi:  61241 samples
  Resolved: 233034 samples (skipped 2000)
    ar:  24273
    bn:  40455
    de:  31014
    en:  31014
    tr:  16037
    uk:  29000
    vi:  61241

── 3. Load mT5-base + LoRA ───────────────────────────────────────────


Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 49, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


trainable params: 1,769,472 || all params: 968,342,784 || trainable%: 0.1827

── 4. Projection MLP ─────────────────────────────────────────────────
  Projection params: 1,182,720

── 5. Load English Checkpoint ────────────────────────────────────────
  Loaded projection + LoRA from /kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt
  en: 'English: ' → bos=5413
  de: 'Deutsch:' → bos=22606
  ar: 'العربية:' → bos=17112
  vi: 'Việt:' → bos=4674
  tr: 'Türk:' → bos=13374
  bn: 'বাংলা:' → bos=46010
  uk: 'Укр:' → bos=153111
✅ All forced_bos tokens unique

── 7. Dataset & Balanced DataLoader ──────────────────────────────────
Train images: 46984, Val images: 5221
Train samples: 209891, Val samples: 23143
✅ No image overlap between train and val
  Train language counts:
    ar:  21903
    bn:  36505
    de:  27876
    en:  27876
    tr:  14469
    uk:  26063
    vi:  55199

  Balanced: ~14469 per language per epoch
  Samples per epoch: 101283
  T

Epoch  1/5 | Train: 3.5216 | Val: 2.4840 | GradNorm: 1.637 | lr: 9.05e-05
  Saved best model (val_loss=2.4840)


Epoch  2/5 | Train: 3.0048 | Val: 2.3582 | GradNorm: 1.605 | lr: 6.55e-05
  Saved best model (val_loss=2.3582)

  ── Caption preview [epoch 2] ────────────────────────────
  [ar] PRED: العربية:كلب أسود يركض في الماء
       REF:  العربية:كلب يقفز في الماء
  [ar] PRED: العربية:كلب أسود يركض في الماء
       REF:  العربية:الكلب الاسود يمر عبر الماء
  [bn] PRED: বাংলা:একটি বাদামী কুকুর একটি পাথরের উপর দৌড়াচ্ছে
       REF:  বাংলা:একটি কালো কুকুর পানির কিনারায় দৌড়াচ্ছে ।
  [bn] PRED: বাংলা:একটি বাদামী কুকুর একটি পাথরের উপর দৌড়াচ্ছে
       REF:  বাংলা:একটি কালো কুকুর তার গলার বাধাবুলি নিয়ে সাগরের পানিতে খেলা করছে
  [de] PRED: Deutsch:Ein Mann in einem schwarzen Hemd steht auf einem roten Fenster.
       REF:  Deutsch:Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.
  [de] PRED: Deutsch:Ein Mann in einem schwarzen Hemd und einem schwarzen Hemd spielt mit einem Spielzeug.
       REF:  Deutsch:Ein Mann lächelt einen ausgestopften Löwen an.
  [en] PRED: English: A ma

Epoch  3/5 | Train: 2.8810 | Val: 2.2960 | GradNorm: 1.613 | lr: 3.45e-05
  Saved best model (val_loss=2.2960)


Epoch  4/5 | Train: 2.8181 | Val: 2.2639 | GradNorm: 1.629 | lr: 9.55e-06
  Saved best model (val_loss=2.2639)

  ── Caption preview [epoch 4] ────────────────────────────
  [ar] PRED: العربية:كلب أسود يركض في الماء
       REF:  العربية:كلب يقفز في الماء
  [ar] PRED: العربية:كلب أسود يركض في الماء
       REF:  العربية:الكلب الاسود يمر عبر الماء
  [bn] PRED: বাংলা:একটি কুকুর পানিতে দৌড়াচ্ছে
       REF:  বাংলা:একটি কালো কুকুর পানির কিনারায় দৌড়াচ্ছে ।
  [bn] PRED: বাংলা:একটি কুকুর পানিতে দৌড়াচ্ছে
       REF:  বাংলা:একটি কালো কুকুর তার গলার বাধাবুলি নিয়ে সাগরের পানিতে খেলা করছে
  [de] PRED: Deutsch:Ein Mann in einem schwarzen Hemd steht auf einem roten Fenster.
       REF:  Deutsch:Ein Mann in einem blauen Hemd steht auf einer Leiter und putzt ein Fenster.
  [de] PRED: Deutsch:Ein Mann in einem schwarzen Hemd und einem schwarzen Hemd spielt auf einem Tisch.
       REF:  Deutsch:Ein Mann lächelt einen ausgestopften Löwen an.
  [en] PRED: English: A man in a white shirt is cleaning a bri

Epoch  5/5 | Train: 2.7883 | Val: 2.2554 | GradNorm: 1.636 | lr: 0.00e+00
  Saved best model (val_loss=2.2554)

Multilingual training complete!
  Best val loss: 2.2554
  Checkpoints: /kaggle/working/multilingual_checkpoints/


In [ ]:
from huggingface_hub import snapshot_download
import os

xm3600_path = "/content/drive/MyDrive/mlp_project/xm3600"
# os.makedirs(xm3600_path, exist_ok=True)

# print("Downloading XM3600...")
# snapshot_download(
#     repo_id="floschne/xm3600",
#     repo_type="dataset",
#     local_dir=xm3600_path,
# )

# data_dir = os.path.join(xm3600_path, "data")
# files = os.listdir(data_dir)
files = os.listdir("/root/.cache/huggingface/datasets/floschne___xm3600/data")
print(f"Done! {len(files)} files")

# Check our target languages exist
targets = ["en", "de", "uk", "ar", "tr", "bn", "vi", "nl", "cs", "pl", "he", "hu", "te", "hi", "id", "mi", "sw"]
for lang in targets:
    found = any(f.startswith(f"{lang}-") for f in files)
    print(f"  {lang}: {'pass' if found else 'fail'}")

Done! 36 files
  en: ✅
  de: ✅
  uk: ✅
  ar: ✅
  tr: ✅
  bn: ✅
  vi: ✅
  nl: ✅
  cs: ✅
  pl: ✅
  he: ✅
  hu: ✅
  te: ✅
  hi: ✅
  id: ✅
  mi: ✅
  sw: ✅


In [ ]:
import os
from datasets import load_dataset, Image as HFImage

XM3600_DATA = "/root/.cache/huggingface/datasets/floschne___xm3600/data"

SEEN_LANGS = ["en", "de", "uk", "ar", "tr", "bn", "vi"]
ZERO_SHOT_LANGS = ["nl", "cs", "pl", "he", "hu", "te", "hi", "id", "mi", "sw"]
ALL_LANGS = SEEN_LANGS + ZERO_SHOT_LANGS

# Load images from English split
print("Loading XM3600 images...")
ds_en = load_dataset("parquet", data_files={"en": f"{XM3600_DATA}/en-*.parquet"}, split="en")
ds_en = ds_en.cast_column("image", HFImage())
print(f"  {len(ds_en)} images | columns: {ds_en.column_names}")

image_id_to_idx = {sample["image_id"]: i for i, sample in enumerate(ds_en)}

# Load captions for all 17 languages
print("\nLoading captions...")
lang_refs = {}
for lang in ALL_LANGS:
    ds_lang = load_dataset("parquet", data_files={lang: f"{XM3600_DATA}/{lang}-*.parquet"}, split=lang)
    refs = {}
    for sample in ds_lang:
        refs[sample["image_id"]] = sample["captions"]
    lang_refs[lang] = refs
    print(f"  {lang}: {len(refs)} images, {sum(len(v) for v in refs.values())} captions")

print("\nAll 17 languages loaded")

Loading XM3600 images...
  3600 images | columns: ['image_id', 'image_locale', 'captions', 'captions_tokenized', 'captions_tokenized_lowercase', 'image']

Loading captions...
  en: 3600 images, 7200 captions
  de: 3600 images, 8643 captions
  uk: 3600 images, 7215 captions
  ar: 3600 images, 7367 captions
  tr: 3600 images, 7233 captions
  bn: 3600 images, 3600 captions
  vi: 3600 images, 7350 captions
  nl: 3600 images, 8059 captions
  cs: 3600 images, 7207 captions
  pl: 3600 images, 7141 captions
  he: 3600 images, 7200 captions
  hu: 3600 images, 7216 captions
  te: 3600 images, 7200 captions
  hi: 3600 images, 8503 captions
  id: 3600 images, 7126 captions
  mi: 3600 images, 4732 captions
  sw: 3600 images, 7046 captions

✅ All 17 languages loaded


In [8]:
!zip -r output.zip /kaggle/working/multilingual_checkpoints

  adding: kaggle/working/multilingual_checkpoints/ (stored 0%)
  adding: kaggle/working/multilingual_checkpoints/best_multilingual.pt^C



zip error: Interrupted (aborting)


In [5]:
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType
from tqdm import tqdm
# ── Device ────────────────────────────────────────────────────────────────────
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB")

Device: cpu


In [ ]:
import torch.nn as nn
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from peft import get_peft_model, LoraConfig, TaskType

# PROJECT_PATH = "/content/drive/MyDrive/mlp_project"
CHECKPOINT = "/kaggle/working/multilingual_checkpoints/best_multilingual_bos.pt"
# CHECKPOINT = "/kaggle/input/models/kavitasriram19/blip-2-mt5-multilingual/pytorch/default/1/best_checkpoint.pt"

# mT5 + LoRA
mt5_tokenizer = AutoTokenizer.from_pretrained("google/mt5-base")
mt5_model = AutoModelForSeq2SeqLM.from_pretrained("google/mt5-base").to(device)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16, lora_alpha=32, lora_dropout=0.1,
    target_modules=["q", "v"],
)
mt5_model = get_peft_model(mt5_model, lora_config)

# Projection + Lang embedding
class ProjectionMLP(nn.Module):
    def __init__(self, in_dim=768, out_dim=768):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim),
            nn.GELU(),
            nn.LayerNorm(out_dim),
            nn.Linear(out_dim, out_dim),
        )
    def forward(self, x):
        return self.net(x)

projection = ProjectionMLP().to(device)
lang_embedding = nn.Embedding(7, 768).to(device)

# Load checkpoint
ckpt = torch.load(CHECKPOINT, weights_only=True, map_location=device)
projection.load_state_dict(ckpt["projection"])
mt5_model.load_state_dict(ckpt["lora"])
# lang_embedding.load_state_dict(ckpt["lang_embedding"])

projection.eval()
mt5_model.eval()
# lang_embedding.eval()

print(f"Model loaded from {CHECKPOINT}")

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 117, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 96, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 72, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

Loading weights:   0%|          | 0/284 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


✅ Model loaded from /kaggle/working/multilingual_checkpoints/best_multilingual_bos.pt


In [ ]:
import gc
import torch
from transformers import Blip2Processor, Blip2Model
from tqdm import tqdm

# device = torch.device("cuda")
FEATURE_FILE = "/kaggle/input/datasets/kavitasriram19/xm3600/xm3600_features.pt"

if os.path.exists(FEATURE_FILE):
    print("Found cached features, loading...")
    saved = torch.load(FEATURE_FILE, weights_only=False)
    xm_features = saved["features"]
    xm_image_ids = saved["image_ids"]
    print(f"  {len(xm_features)} features loaded")
else:
    print("Loading BLIP-2...")
    blip2_processor = Blip2Processor.from_pretrained("Salesforce/blip2-opt-2.7b")
    blip2_model = Blip2Model.from_pretrained(
        "Salesforce/blip2-opt-2.7b", torch_dtype=torch.float16, device_map="auto",
    )
    for p in blip2_model.parameters():
        p.requires_grad = False
    blip2_model.eval()

    n_images = len(ds_en)
    images = [ds_en[i]["image"] for i in range(n_images)]
    xm_image_ids = [ds_en[i]["image_id"] for i in range(n_images)]
    all_feats = []

    for i in tqdm(range(0, n_images, 32), desc="Extracting XM3600 features"):
        batch_imgs = images[i : i + 32]
        inputs = blip2_processor(images=batch_imgs, return_tensors="pt").to(device)
        with torch.no_grad():
            vision_out = blip2_model.vision_model(pixel_values=inputs.pixel_values.half(), return_dict=True)
            image_embeds = vision_out.last_hidden_state
            image_attn = torch.ones(image_embeds.shape[:-1], dtype=torch.long, device=device)
            query_tokens = blip2_model.query_tokens.expand(image_embeds.shape[0], -1, -1)
            qformer_out = blip2_model.qformer(
                query_embeds=query_tokens, encoder_hidden_states=image_embeds,
                encoder_attention_mask=image_attn, return_dict=True,
            )
            all_feats.append(qformer_out.last_hidden_state.cpu().float())

    xm_features = torch.cat(all_feats, dim=0)
    print(f"  Features: {xm_features.shape}")

    torch.save({"features": xm_features, "image_ids": xm_image_ids}, FEATURE_FILE)
    print(f"  Saved to Drive")

    del blip2_model, blip2_processor
    gc.collect()
    torch.cuda.empty_cache()

feat_id_to_idx = {iid: i for i, iid in enumerate(xm_image_ids)}
print(f"XM3600 features ready: {len(xm_features)}")

Found cached features, loading...
  3600 features loaded
✅ XM3600 features ready: 3600


In [25]:
ZERO_SHOT_PREFIXES = {
    "nl": "Nederlands:",    # 27976
    "cs": "Česky:",         # 13528
    "pl": "Polski:",        # 23911
    "he": "עב:",            # 72493  — was broken, fixed
    "hu": "Magyar:",        # 33293
    "te": "తెలుగు:",        # 39971
    "hi": "हिंदी:",         # 50315
    "id": "Indonesia:",     # 3664
    "mi": "Māori:",         # 27205
    "sw": "Kiswah:",        # 35841  — was broken, fixed
}

In [12]:
print("── Zero-shot forced_bos verification ─────────────────────────────")
print(f"{'Lang':<6} {'Prefix':<20} {'IDs':<30} {'First ID':>10} {'Unique':>8}")
print("─" * 75)

# Collect all seen language bos tokens to check for cross-set collisions
seen_bos_tokens = set()
for lang in SEEN_LANGS:
    # seen langs use lang_embedding, no forced_bos — no collision risk
    pass

zero_shot_bos = {}
has_collision = False

for lang, prefix in ZERO_SHOT_PREFIXES.items():
    ids = mt5_tokenizer(prefix, add_special_tokens=False).input_ids
    tokens = mt5_tokenizer.convert_ids_to_tokens(ids)
    first = ids[0]
    unique = (first != 259) and (first not in zero_shot_bos.values())
    status = "✓" if unique else "✗ COLLISION/BROKEN"
    if not unique:
        has_collision = True
    zero_shot_bos[lang] = first
    print(f"{status} {lang:<5} {prefix:<20} {str(ids):<30} {first:>10} {'✓' if unique else '✗'}")

print()
if has_collision:
    print("✗ Some languages have broken or colliding forced_bos tokens — fix before eval")
else:
    print("✓ All zero-shot languages have unique forced_bos tokens — safe to proceed")

print(f"\nZero-shot bos map: {zero_shot_bos}")

── Zero-shot forced_bos verification ─────────────────────────────
Lang   Prefix               IDs                              First ID   Unique
───────────────────────────────────────────────────────────────────────────
✓ nl    Nederlands:          [27976, 267]                        27976 ✓
✓ cs    Česky:               [13528, 276, 267]                   13528 ✓
✓ pl    Polski:              [23911, 267]                        23911 ✓
✓ he    עב:                  [72493, 267]                        72493 ✓
✓ hu    Magyar:              [33293, 267]                        33293 ✓
✓ te    తెలుగు:              [39971, 267]                        39971 ✓
✓ hi    हिंदी:               [50315, 1304, 267]                  50315 ✓
✓ id    Indonesia:           [3664, 267]                          3664 ✓
✓ mi    Māori:               [27205, 2972, 267]                  27205 ✓
✓ sw    Kiswah:              [35841, 25782, 267]                 35841 ✓

✓ All zero-shot languages have unique forced_bo

In [26]:
!pip install pycocoevalcap -q

In [ ]:
from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.bleu.bleu import Bleu
from tqdm import tqdm

# ── Language config ────────────────────────────────────────────────────────────
LANG_TO_ID = {"en": 0, "de": 1, "uk": 2, "ar": 3, "tr": 4, "bn": 5, "vi": 6}

# No proxies — zero-shot uses zero lang_embedding + verified forced_bos
# zero_shot_bos was built in the verification cell above

def encode_seen(features, lang_id_int):
    """Seen languages — learned lang_embedding prepended."""
    projected = projection(features)
    lang_ids  = torch.full((features.shape[0],), lang_id_int, dtype=torch.long, device=device)
    lang_emb  = lang_embedding(lang_ids).unsqueeze(1)
    return torch.cat([lang_emb, projected], dim=1)

def encode_zero_shot(features):
    """Zero-shot languages — zero vector prepended to preserve input shape."""
    projected    = projection(features)
    emb_dim      = lang_embedding.embedding_dim
    zero_lang    = torch.zeros(features.shape[0], 1, emb_dim, device=device)
    return torch.cat([zero_lang, projected], dim=1)

# ── Eval ───────────────────────────────────────────────────────────────────────
eval_image_ids = sorted(image_id_to_idx.keys())
EVAL_BATCH     = 8
results        = {}

for lang in ALL_LANGS:
    is_seen = lang in SEEN_LANGS

    if is_seen:
        lang_id_int = LANG_TO_ID[lang]
        forced_bos  = None
    else:
        # No proxy — zero lang embedding + unique forced_bos
        assert lang in zero_shot_bos, \
            f"{lang} not in zero_shot_bos — run verification cell first"
        forced_bos = zero_shot_bos[lang]

    refs_for_lang = lang_refs[lang]
    preds_dict    = {}
    refs_dict     = {}

    for start in tqdm(range(0, len(eval_image_ids), EVAL_BATCH),
                      desc=f"{lang}", leave=False):
        batch_ids = eval_image_ids[start : start + EVAL_BATCH]
        feat_idx  = [feat_id_to_idx[iid] for iid in batch_ids]
        features  = torch.stack([xm_features[i] for i in feat_idx]).to(device)

        with torch.no_grad():
            if is_seen:
                # encoder_input = encode_seen(features, lang_id_int)
                encoder_input = projection(features)
                gen_kwargs    = {
                    "inputs_embeds": encoder_input,
                    "max_new_tokens": 50,
                    "forced_bos_token_id": LANG_BOS[lang],
                    "num_beams": 4,
                }
            else:
                # encoder_input = encode_zero_shot(features)
                encoder_input = projection(features)
                gen_kwargs    = {
                    "inputs_embeds":    encoder_input,
                    "forced_bos_token_id": forced_bos,
                    "max_new_tokens":   50,
                    "num_beams":        4,
                }
            out = mt5_model.generate(**gen_kwargs)

        for k, iid in enumerate(batch_ids):
            pred = mt5_tokenizer.decode(out[k], skip_special_tokens=True)
            refs = refs_for_lang.get(iid, [])
            if not refs:
                continue
            g             = start + k
            preds_dict[g] = [pred]
            refs_dict[g]  = refs

    cider_score, _ = Cider().compute_score(refs_dict, preds_dict)
    bleu_scores, _ = Bleu(4).compute_score(refs_dict, preds_dict)

    tag = "seen" if is_seen else "zero-shot (forced_bos)"
    results[lang] = {
        "cider":      round(cider_score * 100, 2),
        "bleu1":      round(bleu_scores[0] * 100, 2),
        "bleu4":      round(bleu_scores[3] * 100, 2),
        "type":       tag,
        "forced_bos": None if is_seen else forced_bos,
    }
    print(f"  {lang}: CIDEr={results[lang]['cider']:.2f} | "
          f"BLEU-1={results[lang]['bleu1']:.2f} | "
          f"BLEU-4={results[lang]['bleu4']:.2f} | {tag}")

# ── Summary ────────────────────────────────────────────────────────────────────
print("\n" + "="*65)
print(f"{'Lang':<6} {'CIDEr':>7} {'BLEU-1':>7} {'BLEU-4':>7}  {'Type'}")
print("-"*65)
for lang in ALL_LANGS:
    r = results[lang]
    print(f"{lang:<6} {r['cider']:>7.2f} {r['bleu1']:>7.2f} {r['bleu4']:>7.2f}  {r['type']}")

print("\nEvaluation complete for all 17 languages")

{'testlen': 48132, 'reflen': 37273, 'guess': [48132, 44532, 40932, 37332], 'correct': [7918, 1965, 562, 178]}
ratio: 1.2913368926568485
  en: CIDEr=9.71 | BLEU-1=16.45 | BLEU-4=2.63 | seen


{'testlen': 44961, 'reflen': 40552, 'guess': [44961, 41361, 37761, 34161], 'correct': [4192, 713, 60, 3]}
ratio: 1.1087246005128943
  de: CIDEr=3.01 | BLEU-1=9.32 | BLEU-4=0.39 | seen


{'testlen': 43711, 'reflen': 36223, 'guess': [43711, 40111, 36511, 32911], 'correct': [1622, 95, 3, 1]}
ratio: 1.206719487618331
  uk: CIDEr=0.91 | BLEU-1=3.71 | BLEU-4=0.07 | seen


{'testlen': 35999, 'reflen': 28698, 'guess': [35999, 32399, 28799, 25199], 'correct': [1029, 116, 6, 0]}
ratio: 1.2544079726809794
  ar: CIDEr=2.13 | BLEU-1=2.86 | BLEU-4=0.00 | seen


{'testlen': 41409, 'reflen': 33295, 'guess': [41409, 37809, 34209, 30612], 'correct': [1525, 213, 25, 4]}
ratio: 1.2437002552935503
  tr: CIDEr=1.57 | BLEU-1=3.68 | BLEU-4=0.21 | seen


...

{'testlen': 40141, 'reflen': 31953, 'guess': [40141, 36541, 32941, 29343], 'correct': [34, 0, 0, 0]}
ratio: 1.256251369198471
  hu: CIDEr=0.03 | BLEU-1=0.08 | BLEU-4=0.00 | zero-shot (forced_bos)


{'testlen': 46179, 'reflen': 28365, 'guess': [46179, 42579, 38979, 35379], 'correct': [0, 0, 0, 0]}
ratio: 1.6280274986778906
  te: CIDEr=0.00 | BLEU-1=0.00 | BLEU-4=0.00 | zero-shot (forced_bos)


{'testlen': 41279, 'reflen': 44358, 'guess': [41279, 37679, 34079, 30481], 'correct': [0, 0, 0, 0]}
ratio: 0.9305874926732286
  hi: CIDEr=0.00 | BLEU-1=0.00 | BLEU-4=0.00 | zero-shot (forced_bos)


{'testlen': 41755, 'reflen': 45866, 'guess': [41755, 38155, 34555, 30956], 'correct': [12, 0, 0, 0]}
ratio: 0.910369336763596
  id: CIDEr=0.02 | BLEU-1=0.03 | BLEU-4=0.00 | zero-shot (forced_bos)


KeyboardInterrupt: 

In [29]:
# Cell 1 — Find Azerbaijani forced_bos
candidates = {
    "az_option1": "Azərbaycan:",
    "az_option2": "Azərb:",
    "az_option3": "Azerbaycan:",
    "az_option4": "Azeri:",
    "az_option5": "AZ:",
    "az_option6": "Azərbaycanca:",
}

for label, prefix in candidates.items():
    ids = mt5_tokenizer(prefix, add_special_tokens=False).input_ids
    tokens = mt5_tokenizer.convert_ids_to_tokens(ids)
    first = ids[0]
    unique = first != 259
    print(f"{'✓' if unique else '✗'} {label:<22} '{prefix}' → ids={ids} → first={first}")

✓ az_option1             'Azərbaycan:' → ids=[6948, 267] → first=6948
✗ az_option2             'Azərb:' → ids=[259, 169739, 316, 267] → first=259
✗ az_option3             'Azerbaycan:' → ids=[259, 172433, 267] → first=259
✓ az_option4             'Azeri:' → ids=[203789, 266, 267] → first=203789
✓ az_option5             'AZ:' → ids=[30124, 267] → first=30124
✗ az_option6             'Azərbaycanca:' → ids=[259, 231998, 267] → first=259


In [ ]:
import pandas as pd

# Load Azerbaijani CSV
az_df   = pd.read_csv("/kaggle/input/datasets/kavitasriram19/xm3600-azerbaijani/xm3600_azerbaijani_full_checked.csv")
az_refs = {row["image_id"]: [row["az"]] for _, row in az_df.iterrows()}
print(f"Azerbaijani references loaded: {len(az_refs)} images")

# Fill this in after running Cell 1
AZ_PREFIX = "Azərbaycan:"   # update based on Cell 1 output
az_bos    = mt5_tokenizer(AZ_PREFIX, add_special_tokens=False).input_ids[0]
print(f"Azerbaijani forced_bos: {az_bos} = '{mt5_tokenizer.convert_ids_to_tokens([az_bos])[0]}'")
assert az_bos != 259, "Still hitting 259 — pick a different prefix"

# Eval
preds_dict = {}
refs_dict  = {}

for start in tqdm(range(0, len(eval_image_ids), EVAL_BATCH), desc="az", leave=False):
    batch_ids = eval_image_ids[start : start + EVAL_BATCH]
    feat_idx  = [feat_id_to_idx[iid] for iid in batch_ids]
    features  = torch.stack([xm_features[i] for i in feat_idx]).to(device)

    with torch.no_grad():
        encoder_input = encode_zero_shot(features)
        out = mt5_model.generate(
            inputs_embeds=encoder_input,
            forced_bos_token_id=az_bos,
            max_new_tokens=50,
            num_beams=4,
        )

    for k, iid in enumerate(batch_ids):
        pred = mt5_tokenizer.decode(out[k], skip_special_tokens=True)
        refs = az_refs.get(iid, [])
        if not refs:
            continue
        g             = start + k
        preds_dict[g] = [pred]
        refs_dict[g]  = refs

cider_score, _ = Cider().compute_score(refs_dict, preds_dict)
bleu_scores, _ = Bleu(4).compute_score(refs_dict, preds_dict)

print(f"\nAzerbaijani (zero-shot):")
print(f"  CIDEr={round(cider_score*100,2):.2f} | BLEU-1={round(bleu_scores[0]*100,2):.2f} | BLEU-4={round(bleu_scores[3]*100,2):.2f}")
print(f"  n={len(preds_dict)} images evaluated")

In [33]:
# Qualitative Hindi generation — zero-shot
import requests
from PIL import Image
from io import BytesIO
from transformers import Blip2Processor, Blip2Model
import gc

# Use a few XM3600 image IDs that have English references we can compare against
SAMPLE_IDS = list(image_id_to_idx.keys())[:5]

hi_bos = zero_shot_bos["hi"]  # 50315
print(f"Hindi forced_bos: {hi_bos} = '{mt5_tokenizer.convert_ids_to_tokens([hi_bos])[0]}'")
print()

for iid in SAMPLE_IDS:
    feat_idx = feat_id_to_idx[iid]
    features = xm_features[feat_idx].unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_input = encode_zero_shot(features)
        out = mt5_model.generate(
            inputs_embeds=encoder_input,
            forced_bos_token_id=hi_bos,
            max_new_tokens=50,
            num_beams=4,
        )

    pred_hi = mt5_tokenizer.decode(out[0], skip_special_tokens=True)
    ref_en  = lang_refs["en"].get(iid, ["?"])[0]
    print(f"Image: {iid}")
    print(f"  EN ref:  {ref_en}")
    print(f"  HI pred: {pred_hi}")
    print()

Hindi forced_bos: 50315 = '▁हिंद'

Image: 000411001ff7dd4f
  EN ref:  A rooster and hens surrounded by green leaves.
  HI pred: हिंदh: rất nhiều cây xanh được đặt trên những cây xanh

Image: 0004886b7d043cfd
  EN ref:  The golden items on the table.
  HI pred: हिंदh: một chiếc ghế màu trắng được đặt trên một cái ghế màu trắng

Image: 0035b9006c333719
  EN ref:  A macro shot of a pink flower in a garden.
  HI pred: हिंदh: một cây xanh được treo trên những cây xanh

Image: 00371fbc0d38eab5
  EN ref:  Two stacks of empty eggs tray in a cardboard box.
  HI pred: हिंदh: Các container được đặt trong một cái container

Image: 0046f363bfc646dd
  EN ref:  Six ramekins containing chocolate mousse decorated with raspberries, green leaves and chocolate powder.
  HI pred: हिंदh được đặt trên một chiếc bàn màu trắng được đặt trên một chiếc bàn màu trắng



In [ ]:
# Azerbaijani qualitative examples — zero-shot
SAMPLE_IDS_AZ = list(az_refs.keys())[:5]  # first 5 images that have az references

print(f"Azerbaijani qualitative examples (zero-shot)")
print(f"forced_bos: {az_bos} = '{mt5_tokenizer.convert_ids_to_tokens([az_bos])[0]}'")
print()

for iid in SAMPLE_IDS_AZ:
    if iid not in feat_id_to_idx:
        continue
    feat_idx = feat_id_to_idx[iid]
    features = xm_features[feat_idx].unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_input = encode_zero_shot(features)
        out = mt5_model.generate(
            inputs_embeds=encoder_input,
            forced_bos_token_id=az_bos,
            max_new_tokens=50,
            num_beams=4,
        )

    pred_az = mt5_tokenizer.decode(out[0], skip_special_tokens=True)
    ref_az  = az_refs.get(iid, ["?"])[0]
    ref_en  = lang_refs["en"].get(iid, ["?"])[0]

    print(f"Image: {iid}")
    print(f"  EN ref:  {ref_en}")
    print(f"  AZ ref:  {ref_az}")
    print(f"  AZ pred: {pred_az}")
    print()